In [1]:
from config import CONFIG
from modes import build_runtime_mode_by_name
from workloads import build_runtime_workload_by_name
from model_loader import load_model_for_mode, unload_model
from runner import run_single_benchmark

In [ ]:
# from huggingface_hub import login
# login(new_session=True)

In [ ]:
# from huggingface_hub import whoami
# print(whoami())

{'type': 'user', 'id': '695f984390fca78c745ec290', 'name': 'Aman-Sunesh', 'fullname': 'Aman Sunesh', 'email': 'as18181@nyu.edu', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1777593600, 'isPro': False, 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/noauth/fpZG3qyPrGgXiW-TopC8U.jpeg', 'orgs': [{'type': 'org', 'id': '691d8e884bbe8df0d99462e2', 'name': 'newyorkuniversity', 'fullname': 'New York University', 'email': 'islam.m@nyu.edu', 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1777593600, 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/68e396f2b5bb631e9b2fac9a/orNHmPzOQf2_F5UgXPsu5.png', 'roleInOrg': 'write'}], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'ModeSwitch-LLM', 'role': 'read', 'createdAt': '2026-04-11T22:53:44.893Z'}}}


In [2]:
from modes import get_all_runtime_modes
from model_loader import load_model_for_mode, unload_model

for mode in get_all_runtime_modes(enabled_only=True):
    print("Testing load:", mode.name, mode.runtime_kwargs)
    bundle = None
    try:
        bundle = load_model_for_mode(mode)
        print("  OK")
    except Exception as e:
        print("  FAILED:", e)
    finally:
        unload_model(bundle)

Testing load: fp16_baseline {'dtype': 'float16'}
  FAILED: vllm is not installed, so the vLLM backend cannot be used.
Testing load: int8_quant {'quantization': 'int8'}
  FAILED: vllm is not installed, so the vLLM backend cannot be used.
Testing load: awq_4bit {'quantization': 'awq'}
  FAILED: vllm is not installed, so the vLLM backend cannot be used.
Testing load: prefix_caching {'enable_prefix_caching': True}
  FAILED: vllm is not installed, so the vLLM backend cannot be used.
Testing load: chunked_prefill {'enable_chunked_prefill': True}
  FAILED: vllm is not installed, so the vLLM backend cannot be used.


In [4]:
print("Model:", CONFIG.model.model_name_or_path)
print("Backend:", CONFIG.model.backend)
print("Device:", CONFIG.model.device)
print("Baseline dtype:", CONFIG.model.baseline_dtype)
print("Trials:", CONFIG.system.num_trials)

Model: meta-llama/Llama-3.1-8B-Instruct
Backend: transformers
Device: cuda
Baseline dtype: float16
Trials: 3


In [5]:
runtime_mode = build_runtime_mode_by_name("fp16_baseline")
runtime_workload = build_runtime_workload_by_name("short_prompt_short_output")

print("Mode:")
print(runtime_mode)

print("\nWorkload:")
print(runtime_workload.name)
print("Max new tokens:", runtime_workload.max_new_tokens)
print("Prompt preview:")
print(runtime_workload.prompt[:300])

Mode:
RuntimeMode(name='fp16_baseline', description='FP16 baseline inference mode', backend='transformers', dtype='float16', quantization=None, primary_phase='both', notes='standard mode', speculative_decoding=False, kv_cache_compression=False, prefix_caching=False, chunked_prefill=False, continuous_batching=False, cuda_graphs=False, runtime_kwargs={'torch_dtype': 'float16'})

Workload:
short_prompt_short_output
Max new tokens: 32
Prompt preview:
You are a helpful assistant. Answer the following question clearly and concisely:

Explain the importance of efficient inference for large language models.

You are a helpful assistant. Answer the following question clearly and concisely:

Explain the importance of efficient inference for large lang


In [6]:
bundle = None

try:
    bundle = load_model_for_mode(runtime_mode)
    print("Model loaded successfully.")
    print("Mode name:", bundle.mode_name)
    print("Backend:", bundle.backend)
    print("Tokenizer type:", type(bundle.tokenizer))
    print("Model type:", type(bundle.model))
except Exception as e:
    print("LOAD FAILED:")
    print(type(e).__name__, str(e))
finally:
    unload_model(bundle)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Model loaded successfully.
Mode name: fp16_baseline
Backend: transformers
Tokenizer type: <class 'transformers.tokenization_utils_fast.PreTrainedTokenizerFast'>
Model type: <class 'transformers.models.llama.modeling_llama.LlamaForCausalLM'>


In [7]:
result = run_single_benchmark(
    runtime_mode=runtime_mode,
    workload=runtime_workload,
    trial_index=0,
)

print(result)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


BenchmarkResult(mode_name='fp16_baseline', workload_name='short_prompt_short_output', backend='transformers', trial_index=0, prompt_tokens_target=128, max_new_tokens=32, repeated_prefix=False, memory_pressure=False, start_time_s=288293.8982577, first_token_time_s=288300.7512765, end_time_s=288324.3625853, token_timestamps_s=[288300.7512765, 288301.5138338, 288303.0476176, 288305.2574132, 288305.9894142, 288306.7144419, 288307.4756849, 288308.2455502, 288308.994992, 288309.7617874, 288310.5254626, 288311.2925938, 288312.0658001, 288312.8357653, 288313.6105212, 288314.3466873, 288315.100111, 288315.8363881, 288316.5605288, 288317.316614, 288318.0876731, 288318.821319, 288319.6039415, 288320.4086586, 288322.0130984, 288322.7658704, 288323.5736365, 288324.3616637, 288324.3625572], output_text='Efficient inference is crucial for large language models as it enables them to process and respond to user queries quickly and accurately. Large language models are', output_tokens_generated=28, ttft

In [8]:
result_dict = result.to_dict()

for k, v in result_dict.items():
    print(f"{k}: {v}")

mode_name: fp16_baseline
workload_name: short_prompt_short_output
backend: transformers
trial_index: 0
prompt_tokens_target: 128
max_new_tokens: 32
repeated_prefix: False
memory_pressure: False
start_time_s: 288293.8982577
first_token_time_s: 288300.7512765
end_time_s: 288324.3625853
token_timestamps_s: [288300.7512765, 288301.5138338, 288303.0476176, 288305.2574132, 288305.9894142, 288306.7144419, 288307.4756849, 288308.2455502, 288308.994992, 288309.7617874, 288310.5254626, 288311.2925938, 288312.0658001, 288312.8357653, 288313.6105212, 288314.3466873, 288315.100111, 288315.8363881, 288316.5605288, 288317.316614, 288318.0876731, 288318.821319, 288319.6039415, 288320.4086586, 288322.0130984, 288322.7658704, 288323.5736365, 288324.3616637, 288324.3625572]
output_text: Efficient inference is crucial for large language models as it enables them to process and respond to user queries quickly and accurately. Large language models are
output_tokens_generated: 28
ttft_ms: 6853.0187999713235


In [9]:
if result.error is not None:
    print("Run failed.")
    print(result.error)
else:
    print("Run succeeded.")
    print("TTFT (ms):", result.ttft_ms)
    print("Avg TBT (ms):", result.avg_tbt_ms)
    print("Total latency (ms):", result.total_latency_ms)
    print("Peak GPU memory (MB):", result.peak_gpu_memory_mb)
    print("Output tokens:", result.output_tokens_generated)
    print("Output text preview:")
    print((result.output_text or "")[:500])

Run succeeded.
TTFT (ms): 6853.0187999713235
Avg TBT (ms): 843.2600250006154
Total latency (ms): 30464.327599969693
Peak GPU memory (MB): 15372.90673828125
Output tokens: 28
Output text preview:
Efficient inference is crucial for large language models as it enables them to process and respond to user queries quickly and accurately. Large language models are
